In [38]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *

In [39]:
def find_folders_with_output(directory):
    folders_with_output = []
    for root, dirs, files in os.walk(directory):
        if 'output' in dirs:
            folders_with_output.append(root.split('\\')[1])
            dirs.remove('output')
    return folders_with_output

main_directory = 'model_output'
models = find_folders_with_output(main_directory)
models

['Llama-3-8B-Instruct_4_shot', 'Llama-3-8B-Instruct_4_shot_temp_1']

In [40]:
save = True

eval_path = f"evaluate/batch1"
os.makedirs(eval_path, exist_ok=True)

### Model Output to nice JSON and Failure 

In [41]:
def process_files(model, save=save):
    input_dir = f"model_output/{model}/output/"
    output_dir = f"model_output/{model}/formatted/"
    failure_dir = f"model_output/{model}/failed/"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)
    files = os.listdir(input_dir)
    len_files = len(files)
    for filename in files:
        if filename.endswith('.json'):
            input_file_path = os.path.join(input_dir, filename)
            output_file_path = os.path.join(output_dir, filename)
            failure_file_path = os.path.join(failure_dir, filename)
            try:
                file = read_json(input_file_path)
                print(f"Processing file: {filename}")
                if save:
                    save_json_to_file(file, output_file_path)
            except Exception as e:
                print(f"Error processing file {filename}: {e}")
                if save:
                    shutil.move(input_file_path, failure_file_path)
    
    return len_files

In [42]:
def extract_nct_number(filename):
    parts = filename.split('_')
    nct_number = None
    file_type = None
    for part in parts:
        if part.startswith("NCT"):
            nct_number = part
        if part in ["inc", "exc"]:
            file_type = part
    return nct_number+"_"+file_type


def extract_logical_structure(data):
    structure = defaultdict(int)
    def traverse(node, depth=0):
        nonlocal structure
        structure["depth"] = max(structure["depth"], depth)

        if isinstance(node, dict):
            for key in node:
                if key in ["AND", "OR", "NOT"]:
                    structure[key] += 1
                traverse(node[key], depth + 1)
        elif isinstance(node, list):
            for item in node:
                traverse(item, depth + 1)

    traverse(data)
    return dict(structure)

def extract_raw_texts(data):
    raw_texts = set()
    def traverse(node):
        if isinstance(node, dict):
            for key, value in node.items():
                if key == "raw_text":
                    raw_texts.add(value)
                else:
                    traverse(value)
        elif isinstance(node, list):
            for item in node:
                traverse(item)

    traverse(data)
    return raw_texts

def extract_words(raw_texts):
    words = set()
    for text in raw_texts:
        words.update(text.split())
    return words

In [43]:
for model in models:
    print(model)
    # 1 - Preprocess Files
    anzahl_files = process_files(model, save=save)
    print(anzahl_files)
    # 2 - Evaluate Files
    p2_label_path = "chia_label/p2"
    ready_path = f"model_output/{model}/ready"
    failed_model_path = f"model_output/{model}/failed_inner"
    p2_model_formatted_path = f"model_output/{model}/formatted"
    
    for path in [ready_path, failed_model_path, p2_model_formatted_path]:
        os.makedirs(path, exist_ok=True)
    
    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
    model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}
    
    common_ncts = set(label_files.keys()).intersection(model_files.keys())
    labels = []
    predictions = []
    success_data = []
    
    for nct in common_ncts:
        try:
            label_data = read_json(label_files[nct])
            model_data = read_json(model_files[nct])
            label_structure = extract_logical_structure(label_data)
            model_structure = extract_logical_structure(model_data)
    
            # Try count words
            label_raw_texts = extract_raw_texts(label_data)
            model_raw_texts = extract_raw_texts(model_data)
            
            missing_texts = label_raw_texts - model_raw_texts
            
            label_words = extract_words(label_raw_texts)
            model_words = extract_words(model_raw_texts)
            missing_words = label_words - model_words
            missing_words_pct = len(missing_words) / len(label_words) * 100 if label_words else 0
            print(f"Processing NCT {nct}")
            print(round(missing_words_pct,2), "%")
            print(label_words.issubset(model_words))
            print("Label Text \n")
            print(label_words)
            print("\nModel Text \n")
            print(model_words)
            print()
            
            success_data.append({
                'NCT': nct,
                'label_AND': label_structure.get('AND', 0),
                'label_OR': label_structure.get('OR', 0),
                'label_NOT': label_structure.get('NOT', 0),
                'label_DEPTH': label_structure.get('depth', 0),
                'model_AND': model_structure.get('AND', 0),
                'model_OR': model_structure.get('OR', 0),
                'model_NOT': model_structure.get('NOT', 0),
                'model_DEPTH': model_structure.get('depth', 0),
                'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
                'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
                'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
                'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0
            })
    
            labels.append(label_structure)
            predictions.append(model_structure)
            save and shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
        except Exception as e:
            #print(f"Error processing NCT {nct}: {e}")
            save and shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))
    
    df_success = pd.DataFrame(success_data).set_index('NCT')
    df_success.to_csv(eval_path+f'/{model}_eval.csv')
    true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
    predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values
    
    metrics = {}
    for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
        y_true = true_values[:, i]
        y_pred = predicted_values[:, i]
    
        diffs = y_pred - y_true
        pct_greater = (diffs > 0).sum() / len(diffs) * 100
        pct_less = (diffs < 0).sum() / len(diffs) * 100
        pct_equal = (diffs == 0).sum() / len(diffs) * 100
    
        metrics[metric] = {
            'accuracy': round(accuracy_score(y_true, y_pred), 3),
            'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'pct_greater': round(pct_greater, 2),
            'pct_less': round(pct_less, 2),
            'pct_equal': round(pct_equal, 2)
            # 'confusion_matrix': confusion_matrix(y_true, y_pred)
        }
        metrics_df = pd.DataFrame(metrics).T
        num_nct_files = len(df_success)
        metrics_df['num_nct_files'] = num_nct_files
        metrics_df['model_name'] = model
        # Save Metrics to CSV
        metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))    

Llama-3-8B-Instruct_4_shot
Error processing file Llama-3-8B-Instruct_NCT00050349_exc_4_shot.json: Extra data: line 1 column 2572 (char 2571)
Error processing file Llama-3-8B-Instruct_NCT00050349_inc_4_shot.json: Extra data: line 1 column 1555 (char 1554)
Error processing file Llama-3-8B-Instruct_NCT00061308_exc_4_shot.json: Extra data: line 1 column 1584 (char 1583)
Error processing file Llama-3-8B-Instruct_NCT00061308_inc_4_shot.json: Expecting ',' delimiter: line 1 column 613 (char 612)
Error processing file Llama-3-8B-Instruct_NCT00094861_exc_4_shot.json: Extra data: line 1 column 3196 (char 3195)
Error processing file Llama-3-8B-Instruct_NCT00094861_inc_4_shot.json: Extra data: line 1 column 2316 (char 2315)
Error processing file Llama-3-8B-Instruct_NCT00122070_exc_4_shot.json: Extra data: line 1 column 1573 (char 1572)
Error processing file Llama-3-8B-Instruct_NCT00122070_inc_4_shot.json: Expecting ',' delimiter: line 1 column 671 (char 670)
Error processing file Llama-3-8B-Instru

KeyError: "None of ['NCT'] are in the columns"

In [44]:
# NCT02935855_inc

In [45]:
df_success = pd.DataFrame(success_data).set_index('NCT')
df_success.to_csv(eval_path+f'/{model}_eval.csv')

df_success

KeyError: "None of ['NCT'] are in the columns"

In [46]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    diffs = y_pred - y_true
    pct_greater = (diffs > 0).sum() / len(diffs) * 100
    pct_less = (diffs < 0).sum() / len(diffs) * 100
    pct_equal = (diffs == 0).sum() / len(diffs) * 100


    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'pct_greater': round(pct_greater, 2),
        'pct_less': round(pct_less, 2),
        'pct_equal': round(pct_equal, 2)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))

print(f"{len(df_success)} Daten mit {model}")
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")

    print(f"  % Greater: {values['pct_greater']}")
    print(f"  % Less: {values['pct_less']}")
    print(f"  % Equal: {values['pct_equal']}")
    print()
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

NameError: name 'df_success' is not defined

In [47]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows

NameError: name 'df_success' is not defined